In [1]:
import requests
import time

In [2]:
%%time

base_url = "https://rest.uniprot.org/uniprotkb/search"
query = "organism_id:9606 AND reviewed:true"
fields = "accession,gene_names,protein_name,cc_subcellular_location"
batch_size = 500
format_type = "tsv"

# Output files
unfiltered_file = "all_human_reviewed.txt"
filtered_file = "strict_cell_membrane_proteins.txt"

# Initial request with cursor-based pagination
params = {
    "query": query,
    "fields": fields,
    "format": format_type,
    "size": batch_size
}

print("Downloading all reviewed human proteins from UniProt...")

# Store all results
all_lines = []
cursor = None
page_count = 0

while True:
    if cursor:
        params["cursor"] = cursor
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    lines = response.text.strip().split("\n")

    if page_count == 0:
        all_lines.extend(lines)  # include header
    else:
        all_lines.extend(lines[1:])  # skip duplicate headers

    page_count += 1
    print(f"Downloaded page {page_count} with {len(lines)-1} entries.")

    # Get next cursor
    next_cursor = response.links.get("next", {}).get("url")
    if not next_cursor:
        break
    # Extract cursor from the URL
    cursor = next_cursor.split("cursor=")[-1].split("&")[0]
    time.sleep(1)  # Be gentle to the server

# Save unfiltered file
with open(unfiltered_file, "w", encoding="utf-8") as f:
    f.write("\n".join(all_lines))
print(f"✅ Saved all reviewed human proteins to '{unfiltered_file}'")

Downloaded page 1 with 500 entries.
Downloaded page 2 with 500 entries.
Downloaded page 3 with 500 entries.
Downloaded page 4 with 500 entries.
Downloaded page 5 with 500 entries.
Downloaded page 6 with 500 entries.
Downloaded page 7 with 500 entries.
Downloaded page 8 with 500 entries.
Downloaded page 9 with 500 entries.
Downloaded page 10 with 500 entries.
Downloaded page 11 with 500 entries.
Downloaded page 12 with 500 entries.
Downloaded page 13 with 500 entries.
Downloaded page 14 with 500 entries.
Downloaded page 15 with 500 entries.
Downloaded page 16 with 500 entries.
Downloaded page 17 with 500 entries.
Downloaded page 18 with 500 entries.
Downloaded page 19 with 500 entries.
Downloaded page 20 with 500 entries.
Downloaded page 21 with 500 entries.
Downloaded page 22 with 500 entries.
Downloaded page 23 with 500 entries.
Downloaded page 24 with 500 entries.
Downloaded page 25 with 500 entries.
Downloaded page 26 with 500 entries.
Downloaded page 27 with 500 entries.
Downloaded

In [3]:
from collections import Counter

file_path = "all_human_reviewed.txt"  # Replace with your file name


location_counter = Counter()
subcellular_location_col = 3  # 0-based index

include_terms = {"plasma membrane", "cell membrane", "cell surface membrane", 
                 "apical membrane", "lateral membrane", "basal membrane", 
                 "peripheral membrane"}
excluded_terms = []

header = all_lines[0]
filtered_lines = [header+'\n']

with open(file_path, "r", encoding="utf-8") as f:
    header = f.readline()  # Skip header
    for line in f:
        fields = line.strip().split("\t")
        if len(fields) <= subcellular_location_col:
            continue
        raw_text = fields[subcellular_location_col]
        #print(raw_text)

        next_text = raw_text.split('SUBCELLULAR LOCATION: ')[1]
        v = next_text.split('. ')

        ismembrane = False
        exclude = False
                    
        for location in v:
            if 'Cytoplasmic side' in location:
                exclude = True
            if 'Note' in location:
                pass
            else:
                term = location.split('{')[0]
                term = term.lower()
                if any(substring in term for substring in include_terms):
                    #print('yes', term)
                    ismembrane = True
                else:
                    if 'membrane' in term:
                        if term not in excluded_terms:
                            excluded_terms.append(term)
        if ismembrane == True:
            if exclude == False:
                filtered_lines.append(line)
                

# prtin terms
#print(len(excluded_terms),excluded_terms)

# Save filtered file
with open(filtered_file, "w", encoding="utf-8") as f:
    for line in filtered_lines:
        f.write(line)
print(f"✅ Saved {len(filtered_lines)-1} strict membrane proteins to '{filtered_file}'")
        
# Example usage:


✅ Saved 3752 strict membrane proteins to 'strict_cell_membrane_proteins.txt'


In [4]:
print(len(excluded_terms))
for element in excluded_terms:
    print(element)

637
membrane 
cell projection, stereocilium membrane 
endosome membrane 
endoplasmic reticulum membrane 
golgi apparatus membrane 
mitochondrion inner membrane 
mitochondrion outer membrane 
mitochondrion intermembrane space 
peroxisome membrane 
cell projection, ruffle membrane 
colocalizes in adipocytes with glut4 at actin-based membranes (by similarity)
nucleus membrane 
membrane, caveola 
endomembrane system
cytoplasmic vesicle, clathrin-coated vesicle membrane
membrane; multi-pass membrane protein.
membrane; single-pass membrane protein
golgi apparatus membrane; single-pass membrane protein
endomembrane system; single-pass membrane protein
membrane raft 
nucleus inner membrane 
endomembrane system 
microsome membrane 
early endosome membrane 
membrane; single-pass type i membrane protein.
postsynaptic density membrane 
endoplasmic reticulum membrane; single-pass type i membrane protein.
golgi apparatus, trans-golgi network membrane 
recycling endosome membrane 
secreted, extracell